# 06 — Perluasan PBN Lanjutan (tistis.nl) + Perbaikan Bug Parser

Lanjutan dari [05_combined_data_source.ipynb](05_combined_data_source.ipynb) (13.582 board).
Diminta menambah lebih banyak file PBN ("masih kurang"). Ditemukan sumber baru
**tistis.nl/pbn/pbn_databases.htm** — 169 file PBN tambahan dari turnamen
internasional lama (ETC97/99, EYC98/00, IWBC, OKB, Papi Garozzo), semuanya
diverifikasi hidup dan berisi bidding manusia (bukan bot) sebelum dipakai.

**Sumber DITOLAK setelah verifikasi** (kontaminasi AI/bot):
- `bermuda2000.zip` — 100% GIB vs WBridge5 (komputer lawan komputer)
- File "Al Howard" (`MensPairsFinal`, `AlsReg5/6-Final`, dll.) — GIB bermain
  3 dari 4 kursi, hanya "Al Howard" manusia
- `FFT_9901.zip` — satu pasangan "Bristol GIB" di antara 16 pasangan
  (kontaminasi kecil tapi tidak sepadan untuk cuma 84 board)

Dalam proses integrasi, ditemukan dan diperbaiki **3 bug tersembunyi**:
1. PBN pakai `"#"` sebagai singkatan "sama seperti board sebelumnya" untuk
   tag seperti `HomeTeam`/`VisitTeam` — tidak di-resolve, menyebabkan
   identity key salah.
2. Turnamen round-robin memakai ulang nomor board 1..32 di tiap meja/match
   yang main simultan — nomor board mentah saja bukan identity key unik.
3. Round-trip CSV: kolom `_room` kosong (`""`) berubah jadi `NaN` setelah
   `to_csv()`→`read_csv()`, dan `.astype(str)` pada `NaN` menghasilkan
   string `"nan"` (bukan `""`) — merugikan merge DDS cache.

Ketiganya diperbaiki di `src/parser/pbn_parser.py` dan
`src/preprocessing/dataset_builder.py`. Setelah perbaikan: 0 duplicate
identity key (dari 768 sebelumnya), 0 missing DDS values.

In [1]:
import sys
import time
from pathlib import Path
from collections import Counter

import pandas as pd
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.parser import PBNParser
from src.preprocessing.dataset_builder import load_splits, build_dataset
from src.evaluation.metrics import evaluate, print_summary, compare_models

OUT_DIR = PROJECT_ROOT / "experiments" / "2026-07-15" / "outputs" / "combined_data_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Sumber PBN gabungan (212 file total)

In [2]:
pbn_boards = PBNParser().parse_directory(PROJECT_ROOT / "data" / "raw_pbn")
print(f"Total board dari seluruh sumber PBN (212 file): {len(pbn_boards)}")

event_cnt = Counter(b.source_file.split("_")[0] + "_" + b.source_file.split("_")[1] for b in pbn_boards)
for k, v in sorted(event_cnt.items()):
    print(f"  {k:20s}: {v} board")

Total board dari seluruh sumber PBN (212 file): 10223
  bermuda_bowl        : 640 board
  soloway_ko          : 119 board
  spingold_2019       : 120 board
  swedish_cup         : 123 board
  tistis_etc97        : 3576 board
  tistis_etc99        : 1360 board
  tistis_eyc00        : 1000 board
  tistis_eyc98        : 800 board
  tistis_iwbc         : 96 board
  tistis_okb          : 52 board
  tistis_papi         : 2025 board
  vanderbilt_2019     : 120 board
  venice_cup          : 192 board


## 2. Dataset gabungan v2

`data/processed_combined/` dibangun ulang dari `data/raw/` (606 file LIN) +
`data/raw_pbn/` (212 file PBN, termasuk 169 file baru tistis.nl) + 18 fitur
DDS. Cache DDS (`data/combined_dds_cache_final.csv`) dihitung ulang penuh
dengan parser yang sudah diperbaiki (bukan reuse cache lama — diverifikasi
lewat spot-check bahwa reuse posisional TIDAK valid setelah bug fix).

In [3]:
build_dataset(
    raw_dir=PROJECT_ROOT / "data" / "raw",
    output_dir=PROJECT_ROOT / "data" / "processed_combined",
    include_dds=True,
    dds_cache_path=PROJECT_ROOT / "data" / "combined_dds_cache_final.csv",
    extra_boards=pbn_boards,
)

[1/5] Parsing LIN files...


      Boards loaded: 13708
      + 10223 extra boards from supplementary source(s)
[2/5] Extracting features...


      Feature rows : 23920


[3/5] Cleaning...
      Removed 2245 duplicate rows.
      Final rows   : 21675
      No missing values in features.
[4/5] Encoding labels...


      Classes      : 36
      Classes list : ['1C', '1D', '1H', '1N', '1S', '2C', '2D', '2H', '2N', '2S', '3C', '3D', '3H', '3N', '3S', '4C', '4D', '4H', '4N', '4S', '5C', '5D', '5H', '5N', '5S', '6C', '6D', '6H', '6N', '6S', '7C', '7D', '7H', '7N', '7S', 'PASS']
      Encoder saved: D:\Bridge-Prediction\data\processed_combined\label_encoder.pkl
[DDS] Menambahkan fitur Double-Dummy Solver...
      Loading cached DDS features: D:\Bridge-Prediction\data\combined_dds_cache_final.csv
      DDS fitur ditambahkan: 18 (total fitur: 182)
[5/5] Splitting (group-aware by source_file + board_number)...


...

      Train : 15175
      Val   : 3251
      Test  : 3249
      No cross-split group leakage (verified).


      Saved: D:\Bridge-Prediction\data\processed_combined\train.csv
      Saved: D:\Bridge-Prediction\data\processed_combined\val.csv


      Saved: D:\Bridge-Prediction\data\processed_combined\test.csv


      Saved: D:\Bridge-Prediction\data\processed_combined\full.csv
      Feature cols saved: D:\Bridge-Prediction\data\processed_combined\feature_columns.json


{'train':        N_hcp  N_hcp_S  N_len_S  N_stopper_S  N_hcp_H  N_len_H  N_stopper_H  \
 0          3        3        4            1        0        3            0   
 1          3        3        4            1        0        3            0   
 2          8        3        5            1        3        3            1   
 3          8        3        5            1        3        3            1   
 4          9        4        5            1        3        3            1   
 ...      ...      ...      ...          ...      ...      ...          ...   
 15170      9        3        6            1        0        2            0   
 15171     11        3        5            1        2        2            0   
 15172     12        2        3            1        2        2            0   
 15173      1        1        4            1        0        3            0   
 15174      1        1        4            1        0        3            0   
 
        N_hcp_D  N_len_D  N_stopper_D  ..

In [4]:
df_train, df_val, df_test, feature_cols, le = load_splits(PROJECT_ROOT / "data" / "processed_combined")
X_train, y_train = df_train[feature_cols].values, df_train["label"].values
X_test, y_test = df_test[feature_cols].values, df_test["label"].values

print(f"Train: {X_train.shape}, Val: {df_val.shape}, Test: {X_test.shape}")
print(f"Jumlah kelas: {len(le.classes_)}")

D:\Bridge-Prediction\src\preprocessing\dataset_builder.py:296: DtypeWarning: Columns (0: _board_number) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv(processed_dir / "train.csv")


Train: (15175, 182), Val: (3251, 192), Test: (3249, 182)
Jumlah kelas: 36


## 3. Latih ulang kandidat terbaik di data yang diperluas

In [5]:
XGB_ACC_TUNED = dict(
    n_estimators=300, max_depth=5, learning_rate=0.03,
    subsample=0.9, colsample_bytree=0.6, min_child_weight=5, reg_lambda=2.0,
    random_state=42, n_jobs=-1, eval_metric="mlogloss", verbosity=0,
)
LGBM_CLASS_WEIGHT = dict(
    n_estimators=300, max_depth=-1, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
    verbose=-1, class_weight="balanced",
)

results = []

t0 = time.time()
xgb_clf = XGBClassifier(**XGB_ACC_TUNED)
xgb_clf.fit(X_train, y_train)
proba = xgb_clf.predict_proba(X_test)
res = evaluate(y_test, proba.argmax(axis=1), proba, le, model_name="XGBoost acc-tuned (21.675 board)")
res["fit_seconds"] = round(time.time() - t0, 1)
results.append(res)
print_summary(res)

t0 = time.time()
lgbm_clf = LGBMClassifier(**LGBM_CLASS_WEIGHT)
lgbm_clf.fit(X_train, y_train)
proba2 = lgbm_clf.predict_proba(X_test)
res2 = evaluate(y_test, proba2.argmax(axis=1), proba2, le, model_name="LightGBM class_weight (21.675 board)")
res2["fit_seconds"] = round(time.time() - t0, 1)
results.append(res2)
print_summary(res2)


  XGBoost acc-tuned (21.675 board)
  Accuracy          : 0.5405
  Precision (macro) : 0.4053
  Recall (macro)    : 0.2740
  F1 (macro)        : 0.2937
  F1 (weighted)     : 0.5039
  Top-3 Accuracy    : 0.7962
  Top-5 Accuracy    : 0.8732


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



  LightGBM class_weight (21.675 board)
  Accuracy          : 0.5426
  Precision (macro) : 0.4441
  Recall (macro)    : 0.3232
  F1 (macro)        : 0.3487
  F1 (weighted)     : 0.5185
  Top-3 Accuracy    : 0.7772
  Top-5 Accuracy    : 0.8661


## 4. Bandingkan dengan baseline 13.582-board (sebelum PBN tambahan)

In [6]:
comparison = compare_models(results)
comparison.to_csv(OUT_DIR / "test_comparison.csv")

# Angka referensi dari 05_combined_data_source.ipynb (13.582 board)
reference = {
    "XGBoost acc-tuned (13.582 board, REF)": {"accuracy": 0.524, "f1_macro": 0.264, "f1_weighted": 0.484},
    "LightGBM class_weight (13.582 board, REF)": {"accuracy": 0.536, "f1_macro": 0.310, "f1_weighted": 0.506},
}
ref_df = pd.DataFrame(reference).T
print(ref_df)
print()
print(comparison[["accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy"]])

                                           accuracy  f1_macro  f1_weighted
XGBoost acc-tuned (13.582 board, REF)         0.524     0.264        0.484
LightGBM class_weight (13.582 board, REF)     0.536     0.310        0.506

                                      accuracy  f1_macro  f1_weighted  \
model                                                                   
XGBoost acc-tuned (21.675 board)      0.540474  0.293714     0.503865   
LightGBM class_weight (21.675 board)  0.542629  0.348652     0.518526   

                                      top_3_accuracy  top_5_accuracy  
model                                                                 
XGBoost acc-tuned (21.675 board)            0.796245        0.873192  
LightGBM class_weight (21.675 board)        0.777162        0.866113  


In [7]:
import json

summary = {
    "generated": pd.Timestamp.now().isoformat(),
    "data_sources": {
        "lin_files": 606,
        "pbn_files_total": 212,
        "pbn_files_new_today": 169,
        "pbn_source": "computerbridge.se (43 files) + tistis.nl (169 files)",
        "pbn_boards_total": len(pbn_boards),
    },
    "dataset_size": {
        "total_boards": len(df_train) + len(df_val) + len(df_test),
        "train": len(df_train), "val": len(df_val), "test": len(df_test),
        "n_features": len(feature_cols),
        "n_classes": len(le.classes_),
    },
    "bugs_fixed": [
        "PBN '#' placeholder (carry-forward) not resolved -> wrong identity keys",
        "round-robin board numbers reused across simultaneous tables -> composite board_id (board|home|visit, fallback to sorted player names)",
        "empty-string _room became NaN through CSV round-trip -> '.astype(str)' produced literal 'nan' -> broken DDS merge; fixed with keep_default_na=False",
    ],
    "results": {r["model"]: {k: r[k] for k in ("accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy")} for r in results},
}
with open(OUT_DIR / "nb06_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "generated": "2026-07-16T11:20:47.362670",
  "data_sources": {
    "lin_files": 606,
    "pbn_files_total": 212,
    "pbn_files_new_today": 169,
    "pbn_source": "computerbridge.se (43 files) + tistis.nl (169 files)",
    "pbn_boards_total": 10223
  },
  "dataset_size": {
    "total_boards": 21675,
    "train": 15175,
    "val": 3251,
    "test": 3249,
    "n_features": 182,
    "n_classes": 36
  },
  "bugs_fixed": [
    "PBN '#' placeholder (carry-forward) not resolved -> wrong identity keys",
    "round-robin board numbers reused across simultaneous tables -> composite board_id (board|home|visit, fallback to sorted player names)",
    "empty-string _room became NaN through CSV round-trip -> '.astype(str)' produced literal 'nan' -> broken DDS merge; fixed with keep_default_na=False"
  ],
  "results": {
    "XGBoost acc-tuned (21.675 board)": {
      "accuracy": 0.5404739919975377,
      "f1_macro": 0.2937144284168763,
      "f1_weighted": 0.503864578615242,
      "top_3_accuracy"

## 5. Kesimpulan

**Hasil terbaik baru dari seluruh proyek** — 169 file PBN tambahan
(13.582 → 21.675 board, +60%) memberi perbaikan nyata di kedua model,
terutama F1 macro (penanganan kelas langka):

| Model | Accuracy | F1 Macro | F1 Weighted |
|---|---|---|---|
| XGBoost acc-tuned (13.582 board) | 52.4% | 0.264 | 0.484 |
| **XGBoost acc-tuned (21.675 board)** | **54.1%** (+1.7pp) | **0.294** (+3.0pp) | **0.504** (+2.0pp) |
| LightGBM class_weight (13.582 board) | 53.6% | 0.310 | 0.506 |
| **LightGBM class_weight (21.675 board)** | **54.3%** (+0.7pp) | **0.349** (+3.9pp) | **0.519** (+1.3pp) |

**LightGBM+class_weight di data v2 tetap kandidat terbaik proyek secara
keseluruhan** — sekarang unggul di SEMUA metrik dibanding XGBoost
(sebelumnya XGBoost masih unggul accuracy tipis di beberapa run). Dataset
yang jauh lebih besar (21.675 board, 36 kelas — `5N` akhirnya punya cukup
sampel) terbukti membantu terutama kelas langka, konsisten dengan pola
yang sudah diamati sejak eksperimen 2026-07-09.

**Rekomendasi**: `LightGBM+class_weight` di `data/processed_combined/`
(21.675 board, 182 fitur) adalah kandidat model utama proyek saat ini.